In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from functools import partial
from lib import wynne
from nystroem_ksd.ksd import StatTest, BM_clip, InfiniteDimKSD, ContextTimer, TestBenchmark, CA_freqs, NystroemInfiniteDimKSD

In [4]:
# Specify number of frequencies to use in numerical approximation of inner products
# Here we use 100 meaning we are working in the space spanned by the first
# 100 basis functions of Brownian motion
n_freqs = 100

# Set the target covariance operator in matrix form with respect to 
# the specified number of basis elements, these are the eigenvalues 
# of Brownian motion decomposition
C = np.diag([(1/((i-0.5)*np.pi))**(2) for i in np.arange(1,n_freqs+1)])

# Set hyperparameters
T_1 = np.eye(n_freqs)
n_adjust_freqs = 50
T_2 = np.eye(n_freqs)
T_2[np.diag_indices(n_adjust_freqs)] = C[np.diag_indices(n_adjust_freqs)]**(-1)

reps = 500

In [5]:
level_alpha=0.05

In [8]:
ns = [50,100,200]
data = { 
    "Ex1" : BM_clip(n_freqs = n_freqs,clip_freq = n_freqs),
    "Ex2" : BM_clip(n_freqs = n_freqs,clip_freq = 8),
    "Ex5" : CA_freqs(n_freqs = n_freqs,a_1=0,a_2=1,a_3=0)
    }

tests = {
    #"KSD_SE_T1" :  { "kernel" : "SE",  "T" : T_1},
    "KSD_SE_T2" :  { "kernel" : "SE",  "T" : T_2},
    #"KSD_IMQ_T1" : { "kernel" : "IMQ", "T" : T_1},
    #"KSD_IMQ_T2" : { "kernel" : "IMQ", "T" : T_2},
    }

res = pd.DataFrame()
for n in tqdm(ns):
    for data_key, data_value in data.items():
        print("Experiment: ", data_key)
        for test_key, test_value in tests.items():
            gamma = InfiniteDimKSD.median_heuristic(data_value.gen(n),T=test_value["T"])
            qt_test = InfiniteDimKSD(cov=C,T=test_value["T"]/gamma,gamma=gamma,kernel_type=test_value["kernel"],level_alpha=level_alpha)
            nys_test = NystroemInfiniteDimKSD(cov=C,T=test_value["T"]/gamma,gamma=gamma,nystroem_samples_func=lambda n : int(np.sqrt(n))*4, kernel_type=test_value["kernel"],level_alpha=level_alpha)
            
            with ContextTimer() as t:
                rejects = TestBenchmark(stat_test=qt_test).error(n=n,data_gen=data_value.gen, repetitions=reps)
            res = pd.concat((res,pd.DataFrame({
                "n" : [n],
                "data" : [data_key],
                "test" : [test_key + "_qt"],
                "rejects" : [rejects],
                "time" : [t.secs]
            })))
            with ContextTimer() as t:
                rejects = TestBenchmark(stat_test=nys_test).error(n=n,data_gen=data_value.gen, repetitions=reps)
            res = pd.concat((res,pd.DataFrame({
                "n" : [n],
                "data" : [data_key],
                "test" : [test_key + "_nys"],
                "rejects" : [rejects],
                "time" : [t.secs]
            })))

  0%|                                                     | 0/3 [00:00<?, ?it/s]

Experiment:  Ex1
Experiment:  Ex2
Experiment:  Ex5


 33%|██████████████▋                             | 1/3 [01:42<03:25, 102.60s/it]

Experiment:  Ex1
Experiment:  Ex2
Experiment:  Ex5


 67%|█████████████████████████████▎              | 2/3 [03:42<01:52, 112.59s/it]

Experiment:  Ex1
Experiment:  Ex2
Experiment:  Ex5


100%|████████████████████████████████████████████| 3/3 [08:10<00:00, 163.35s/it]


In [10]:
res.pivot_table(values="time",index="data",columns="test",aggfunc="max")

test,KSD_SE_T2_nys,KSD_SE_T2_qt
data,,
Ex1,32.808151,47.493662
Ex2,39.808407,66.212375
Ex5,33.317839,48.175709


In [11]:
res.to_csv("../results/infinite-dim-ksd.csv")